In [ ]:
setwd("splicing")

library(plyr)
library(dplyr)
library(qvalue)
library(msigdbr)
library(data.table)

source("code/enrichment_analysis.R")

In [3]:
msigdb <- msigdbr(species="Mus musculus", category="C5")
sets <- tapply(msigdb$gene_symbol, INDEX=msigdb$gs_name, FUN=list)
sets <- sets[grep("^GO", names(sets))]

# CalTRAP NA vs. CalTRAP SR

In [ ]:
experiment_path <- "data/rmats/batch2/caltrap_NA_vs_SR"

In [ ]:
# combine all detected splicing events to get background

all_files <- list.files(path=experiment_path, pattern="fromGTF\\.[^.]+\\.txt")
all_events_list <- lapply(all_files, function(x) {
    fread(file.path(experiment_path, x), data.table=FALSE)
})
all_events_df <- do.call(rbind.fill, all_events_list)
all_genes <- unique(all_events_df$geneSymbol)

In [ ]:
# combine singificant splicing events

signif_files <- list.files(path=experiment_path, pattern="filtered")

signif_events_list <- lapply(signif_files, function(x) {
    df <- fread(file.path(experiment_path, x), data.table=FALSE)
    df$Event_type <- unlist(strsplit(x, split=".", fixed=TRUE))[1]
    df
})
signif_events_df <- do.call(rbind.fill, signif_events_list)
signif_genes <- unique(signif_events_df$geneSymbol)

## All significant events

In [9]:
enrich_df <- get_enrich(signif_genes, all_genes, sets)
head(enrich_df, 20)

,Pval,P.adj
,<dbl>,<dbl>
GOCC_NEURON_PROJECTION,2.812379e-13,2.924593e-09
GOCC_AXON,6.501063e-13,3.380227e-09
GOBP_NEURON_DIFFERENTIATION,1.569591e-12,5.440726e-09
GOMF_RNA_BINDING,5.030566e-12,1.307821e-08
GOCC_SYNAPSE,1.555581e-11,3.235298e-08
GOBP_CELLULAR_COMPONENT_MORPHOGENESIS,2.537648e-11,3.801197e-08
GOBP_CELL_PART_MORPHOGENESIS,2.558744e-11,3.801197e-08
GOBP_NEUROGENESIS,3.075226e-11,3.997410e-08
GOBP_NEURON_DEVELOPMENT,1.559742e-10,1.802195e-07


## Retained introns

### Negative values corresponds to more retained introns in SR samples

In [10]:
RI_cal_df <- signif_events_df[(signif_events_df$Event_type == "RI") & (signif_events_df$IncLevelDifference < 0),]
RI_cal_genes <- unique(RI_cal_df$geneSymbol)
print(length(RI_cal_genes))

[1] 54


In [11]:
enrich_RI_cal_df <- get_enrich(RI_cal_genes, all_genes, sets)
head(enrich_RI_cal_df)

,Pval,P.adj
,<dbl>,<dbl>
GOMF_SULFURTRANSFERASE_ACTIVITY,0.0001566038,0.8121669
GOBP_BONE_MARROW_DEVELOPMENT,0.0002343014,0.8121669
GOMF_TAT_PROTEIN_BINDING,0.0002343014,0.8121669
GOBP_REGULATION_OF_PROTEIN_MODIFICATION_PROCESS,0.0003989931,1.0000000
GOBP_REPLICATIVE_SENESCENCE,0.0010151306,1.0000000
GOBP_FIBROBLAST_PROLIFERATION,0.0011076990,1.0000000


### Positive values corresponds to more retained introns in NA samples

In [12]:
RI_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "RI") & (signif_events_df$IncLevelDifference > 0),]
RI_ctrl_genes <- unique(RI_ctrl_df$geneSymbol)
print(length(RI_ctrl_genes))

[1] 72


In [13]:
enrich_RI_ctrl_df <- get_enrich(RI_ctrl_genes, all_genes, sets)
head(enrich_RI_ctrl_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_RNA_PROCESSING,2.325603e-07,0.002418394
GOBP_MRNA_PROCESSING,1.774957e-06,0.009161033
GOBP_REGULATION_OF_MRNA_METABOLIC_PROCESS,2.642860e-06,0.009161033
GOBP_MRNA_METABOLIC_PROCESS,1.142495e-05,0.029702020
GOMF_RNA_BINDING,2.439234e-05,0.050731191
GOBP_REGULATION_OF_JNK_CASCADE,1.235017e-04,0.214048997


## Skipped exons

### Negative values corresponds to more skipped exons in SR samples

In [14]:
SE_cal_df <- signif_events_df[(signif_events_df$Event_type == "SE") & (signif_events_df$IncLevelDifference < 0),]
SE_cal_genes <- unique(SE_cal_df$geneSymbol)
print(length(SE_cal_genes))

[1] 488


In [15]:
enrich_SE_cal_df <- get_enrich(SE_cal_genes, all_genes, sets)
head(enrich_SE_cal_df, 20)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_CELL_JUNCTION_ORGANIZATION,1.973146e-06,0.02051874
GOMF_CELL_ADHESION_MOLECULE_BINDING,6.066872e-06,0.02356376
GOMF_CADHERIN_BINDING,6.797892e-06,0.02356376
GOCC_NEURON_PROJECTION,9.184959e-06,0.02387860
GOBP_NEGATIVE_REGULATION_OF_CELLULAR_COMPONENT_ORGANIZATION,1.177999e-05,0.02450003
GOCC_NUCLEAR_BODY,3.157598e-05,0.04869346
GOCC_SYNAPSE,3.287638e-05,0.04869346
GOBP_NEURON_DIFFERENTIATION,3.746011e-05,0.04869346
GOBP_POSITIVE_REGULATION_OF_MELANOCYTE_DIFFERENTIATION,4.730088e-05,0.04918819


### Positive values corresponds to more skipped exons in NA samples

In [16]:
SE_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "SE") & (signif_events_df$IncLevelDifference > 0),]
SE_ctrl_genes <- unique(SE_ctrl_df$geneSymbol)
print(length(SE_ctrl_genes))

[1] 596


In [17]:
enrich_SE_ctrl_df <- get_enrich(SE_ctrl_genes, all_genes, sets)
head(enrich_SE_ctrl_df, 10)

,Pval,P.adj
,<dbl>,<dbl>
GOCC_NEURON_PROJECTION,9.234388e-15,9.602840e-11
GOCC_SYNAPSE,2.837994e-13,1.475615e-09
GOCC_AXON,5.866834e-13,2.033640e-09
GOBP_CYTOSKELETON_ORGANIZATION,1.297452e-12,3.373052e-09
GOBP_CELL_JUNCTION_ORGANIZATION,6.770869e-12,1.408205e-08
GOBP_NEURON_DEVELOPMENT,3.666415e-10,6.354509e-07
GOBP_NEURON_DIFFERENTIATION,4.461303e-10,6.627584e-07
GOMF_CYTOSKELETAL_PROTEIN_BINDING,5.376897e-10,6.989294e-07
GOCC_CELL_CORTEX,1.483814e-09,1.714465e-06


## Mutally exclusive exons

In [18]:
MXE_df <- signif_events_df[signif_events_df$Event_type == "MXE",]
MXE_genes <- unique(MXE_df$geneSymbol)
print(length(MXE_genes))

[1] 102


In [19]:
enrich_MXE_df <- get_enrich(MXE_genes, all_genes, sets)
head(enrich_MXE_df)

,Pval,P.adj
,<dbl>,<dbl>
GOMF_RNA_CAP_BINDING,8.831627e-05,0.9184009
GOCC_SARCOLEMMA,5.379105e-04,1.0000000
GOBP_REGULATION_OF_TRANSLATION_AT_SYNAPSE_MODULATING_SYNAPTIC_TRANSMISSION,5.596919e-04,1.0000000
GOMF_ACTIN_BINDING,5.807220e-04,1.0000000
GOCC_I_BAND,6.272199e-04,1.0000000
GOBP_ENSHEATHMENT_OF_NEURONS,7.630551e-04,1.0000000


## Alternative 3' splice site

### Negative values corresponds to longer 3' end in SR samples

In [20]:
A3SS_cal_df <- signif_events_df[(signif_events_df$Event_type == "A3SS") & (signif_events_df$IncLevelDifference < 0),]
A3SS_cal_genes <- unique(A3SS_cal_df$geneSymbol)
print(length(A3SS_cal_genes))

[1] 93


In [21]:
enrich_A3SS_cal_df <- get_enrich(A3SS_cal_genes, all_genes, sets)
head(enrich_A3SS_cal_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_PROTEIN_MODIFICATION_BY_SMALL_PROTEIN_CONJUGATION,0.0002085201,0.7383683
GOBP_CHROMOSOME_ATTACHMENT_TO_THE_NUCLEAR_ENVELOPE,0.0002805377,0.7383683
GOBP_REGULATION_OF_NON_MEMBRANE_SPANNING_PROTEIN_TYROSINE_KINASE_ACTIVITY,0.0002805377,0.7383683
GOBP_PROTEIN_MODIFICATION_BY_SMALL_PROTEIN_CONJUGATION_OR_REMOVAL,0.0002840151,0.7383683
GOBP_PROTEIN_CONTAINING_COMPLEX_ORGANIZATION,0.0003611359,0.7481908
GOBP_POSITIVE_REGULATION_OF_MRNA_METABOLIC_PROCESS,0.0004316901,0.7481908


### Positive values corresponds to longer 3' end in NA samples


In [22]:
A3SS_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "A3SS") & (signif_events_df$IncLevelDifference > 0),]
A3SS_ctrl_genes <- unique(A3SS_ctrl_df$geneSymbol)
print(length(A3SS_ctrl_genes))

[1] 106


In [23]:
enrich_A3SS_ctrl_df <- get_enrich(A3SS_ctrl_genes, all_genes, sets)
head(enrich_A3SS_ctrl_df)

,Pval,P.adj
,<dbl>,<dbl>
GOMF_TRANSCRIPTION_FACTOR_BINDING,9.406519e-07,0.009781839
GOMF_HELICASE_ACTIVITY,5.843013e-06,0.030380746
GOBP_POSITIVE_REGULATION_OF_CELLULAR_BIOSYNTHETIC_PROCESS,1.279886e-05,0.044365118
GOMF_CATALYTIC_ACTIVITY_ACTING_ON_A_NUCLEIC_ACID,2.324798e-05,0.052673811
GOBP_POSITIVE_REGULATION_OF_TRANSCRIPTION_BY_RNA_POLYMERASE_II,3.027062e-05,0.052673811
GOBP_NEGATIVE_REGULATION_OF_TRANSCRIPTION_BY_RNA_POLYMERASE_II,3.039166e-05,0.052673811


## Alternative 5' splice site

### Negative values corresponds to longer 5' end in SR samples

In [24]:
A5SS_cal_df <- signif_events_df[(signif_events_df$Event_type == "A5SS") & (signif_events_df$IncLevelDifference < 0),]
A5SS_cal_genes <- unique(A5SS_cal_df$geneSymbol)
print(length(A5SS_cal_genes))

[1] 58


In [25]:
enrich_A5SS_cal_df <- get_enrich(A5SS_cal_genes, all_genes, sets)
head(enrich_A5SS_cal_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_PROTEIN_MODIFICATION_BY_SMALL_PROTEIN_CONJUGATION,6.611075e-06,0.06874857
GOBP_PROTEIN_MODIFICATION_BY_SMALL_PROTEIN_CONJUGATION_OR_REMOVAL,3.284710e-05,0.17078848
GOCC_CULLIN_RING_UBIQUITIN_LIGASE_COMPLEX,1.441492e-04,0.49966930
GOBP_NEURONAL_ACTION_POTENTIAL_PROPAGATION,2.704354e-04,0.70306441
GOBP_REGULATION_OF_RNA_SPLICING,1.809556e-03,1.00000000
GOCC_AXON_INITIAL_SEGMENT,1.846438e-03,1.00000000


### Positive values corresponds to longer 5' end in NA samples


In [26]:
A5SS_ctrl_df <- signif_events_df[(signif_events_df$Event_type == "A5SS") & (signif_events_df$IncLevelDifference > 0),]
A5SS_ctrl_genes <- unique(A5SS_ctrl_df$geneSymbol)
print(length(A5SS_ctrl_genes))

[1] 72


In [27]:
enrich_A5SS_ctrl_df <- get_enrich(A5SS_ctrl_genes, all_genes, sets)
head(enrich_A5SS_ctrl_df)

,Pval,P.adj
,<dbl>,<dbl>
GOBP_POSITIVE_REGULATION_OF_PROTEIN_SUMOYLATION,0.0009904703,1
GOCC_CD40_RECEPTOR_COMPLEX,0.0012338083,1
GOBP_CELLULAR_RESPONSE_TO_REACTIVE_OXYGEN_SPECIES,0.0015320466,1
GOBP_MUSCLE_HYPERTROPHY_IN_RESPONSE_TO_STRESS,0.0021165181,1
GOBP_NEUROMUSCULAR_PROCESS_CONTROLLING_POSTURE,0.0021165181,1
GOBP_HEMATOPOIETIC_STEM_CELL_HOMEOSTASIS,0.0024607490,1
